In [2]:
## Recover de tabelas no catálogo
!pip install boto3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 2.6 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 1.7 MB/s eta 0:00:0000:0100:01m

[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import boto3
from pyiceberg.catalog import load_catalog
from pyiceberg.exceptions import NamespaceAlreadyExistsError, TableAlreadyExistsError

# --- CONFIGURAÇÕES ---
MINIO_PARAMS = {
    "endpoint_url": "http://minio:9000",
    "aws_access_key_id": "minioadmin",
    "aws_secret_access_key": "minioadmin",
    "region_name": "us-east-1"
}

BUCKET_NAME = "warehouse"
LAYERS = ["bronze", "silver", "gold"]

# Inicializa o Catálogo Iceberg
catalog = load_catalog(
    "dev",
    **{
        "uri": "http://rest:8181",
        "s3.endpoint": MINIO_PARAMS["endpoint_url"],
        "s3.access-key-id": MINIO_PARAMS["aws_access_key_id"],
        "s3.secret-access-key": MINIO_PARAMS["aws_secret_access_key"],
        "s3.region": MINIO_PARAMS["region_name"],
    }
)

# Inicializa o Cliente S3
s3_client = boto3.client("s3", **MINIO_PARAMS)

def get_latest_metadata(layer, table_name):
    """Busca o arquivo .metadata.json mais recente dentro da pasta /metadata/"""
    prefix = f"{layer}/{table_name}/metadata/"
    response = s3_client.list_objects_v2(Bucket=BUCKET_NAME, Prefix=prefix)
    
    if "Contents" not in response:
        return None

    # Filtra apenas arquivos .metadata.json e ordena pelo mais recente
    metadata_files = [
        obj for obj in response["Contents"] 
        if obj["Key"].endswith(".metadata.json")
    ]
    
    if not metadata_files:
        return None
    
    latest_file = max(metadata_files, key=lambda x: x["LastModified"])
    return f"s3://{BUCKET_NAME}/{latest_file['Key']}"

def recover_all():
    for layer in LAYERS:
        print(f"--- Escaneando camada: {layer} ---")
        
        # Cria o namespace se não existir
        try:
            catalog.create_namespace(layer)
        except NamespaceAlreadyExistsError:
            pass

        # Lista "pastas" (tabelas) dentro da camada
        # No S3 não existem pastas reais, então usamos o CommonPrefixes
        paginator = s3_client.get_paginator('list_objects_v2')
        for page in paginator.paginate(Bucket=BUCKET_NAME, Prefix=f"{layer}/", Delimiter='/'):
            for prefix in page.get('CommonPrefixes', []):
                # Extrai o nome da tabela da path (ex: silver/minha_tabela/ -> minha_tabela)
                table_name = prefix['Prefix'].split('/')[-2]
                
                metadata_path = get_latest_metadata(layer, table_name)
                
                if metadata_path:
                    identifier = f"{layer}.{table_name}"
                    try:
                        catalog.register_table(identifier, metadata_path)
                        print(f"✅ {identifier} registrado via: {metadata_path.split('/')[-1]}")
                    except TableAlreadyExistsError:
                        print(f"⚠️ {identifier} já consta no catálogo.")
                    except Exception as e:
                        print(f"❌ Erro ao registrar {identifier}: {e}")
                else:
                    print(f"❓ Ninguém em casa: {layer}/{table_name} não possui metadados válidos.")

if __name__ == "__main__":
    recover_all()

--- Escaneando camada: bronze ---
⚠️ bronze.btcusdt5m já consta no catálogo.
⚠️ bronze.carros_astro já consta no catálogo.
⚠️ bronze.crypto_candles já consta no catálogo.
⚠️ bronze.ingestion já consta no catálogo.
⚠️ bronze.ingestion_messages já consta no catálogo.
⚠️ bronze.nhl_api_calls já consta no catálogo.
⚠️ bronze.nubank_raw já consta no catálogo.
⚠️ bronze.raw_excel já consta no catálogo.
⚠️ bronze.teste_table já consta no catálogo.
--- Escaneando camada: silver ---
⚠️ silver.btcusdt5m já consta no catálogo.
⚠️ silver.carros_astro já consta no catálogo.
⚠️ silver.cripto_currency já consta no catálogo.
⚠️ silver.mapping_nubank_categorias já consta no catálogo.
⚠️ silver.nhl_games já consta no catálogo.
⚠️ silver.nhl_play_by_play já consta no catálogo.
⚠️ silver.nhl_schedule já consta no catálogo.
⚠️ silver.nubank_agg já consta no catálogo.
⚠️ silver.nubank_clean já consta no catálogo.
--- Escaneando camada: gold ---
✅ gold.nhl_teams registrado via: 00000-2213711e-7040-4276-abbe-